In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.794, 10: 0.8025, 20: 0.7945, 30: 0.8020000000000002, 40: 0.804, 50: 0.8109999999999999, 60: 0.8144999999999998, 70: 0.8145, 80: 0.8175000000000002, 90: 0.8220000000000001, 100: 0.8260000000000002, 110: 0.8160000000000002, 120: 0.8159999999999998, 130: 0.8225, 140: 0.8215, 150: 0.8219999999999998, 160: 0.8335000000000001, 170: 0.8290000000000003, 180: 0.8285000000000002, 190: 0.835, 200: 0.8300000000000001, 210: 0.8314999999999999, 220: 0.8345, 230: 0.8374999999999998, 240: 0.842, 250: 0.8440000000000001, 260: 0.8389473684210526, 270: 0.8373684210526317, 280: 0.8389189189189188, 290: 0.8454054054054052, 300: 0.8459459459459461}
{0: 0.016124, 10: 0.01280375, 20: 0.01423975, 30: 0.012055999999999999, 40: 0.010184, 50: 0.010479, 60: 0.010879749999999999, 70: 0.01139975, 80: 0.00938375, 90: 0.009276000000000001, 100: 0.010024, 110: 0.009543999999999999, 120: 0.008364, 130: 0.00790375, 140: 0.00944775, 150: 0.008336, 160: 0.00672775, 170: 0.009139, 180: 0.007917750000000003, 190: 0.005